In [1]:
import math
from typing import Optional , List

import torch
from torch import nn
from labml import tracker

In [9]:
## This module does a linear transformation and splits 
# The vector into given number of heads for multi-head attention 
# This is used to transform key, query, value

class PrepareForMultiHeadAttention(nn.Module):
    
    def __init__(self, d_model: int ,d_k: int, heads: int, bias:bool ):
        super().__init()  ## calls the parent nn.Module constructor to set things up properly
        #Linear layer for linear transform
        self.linear=nn.Linear(d_model, heads*d_k,bias=bias)
        #number of heads 
        self.heads=heads
        #number of dimensions in vector in each head 
        self.d_k=d_k
    





# forward function takes an input x applies a linear projection to it
# and then reshapes the output so that it split accross multiple attnetion heads
# x.shape=(seq_len, batch_size, d_model)
# For example sentence : The cat sat - > 3 tokens
# Batch size : 2 ( processing 2 sentences at once)
# d_model=512 ( each token is represented as a 512 dim vector)

    def forward(self, x:torch.Tensor):
        head_shape=x.shape[:-1]
        x=self.linear(x) #linear transform (ax+b)
        #split last dimension into heads
        x=x.view(*head_shape, self.heads, self.d_k)
        return x 


In [14]:
class MultiHeadAttention(nn.Module):
    def __int__(self, heads:int, d_model:int , dropout_prob:float,bias: bool =True ):
     
     super().__init__()
     self.d_k=d_model//heads
     self.heads=heads
     ## These transforms the query, key and value vectors for multi-head attention
     self.query=PrepareForMultiHeadAttention(d_model, heads,self.d_k, bias=bias )
     self.key=PrepareForMultiHeadAttention(d_model, heads,self.d_k, bias=bias )
     self.value=PrepareForMultiHeadAttention(d_model, heads,self.d_k, bias=bias )

     #apply softmax
     self.softmax=nn.Softmax(dim=1)
     self.output=nn.Linear(d_model, d_model)
     self.dropout=nn.Dropout(dropout_prob)
     #scaling factor before the softmax
     self.scale=1/math.sqrt(self.d_k)
     self.attn=None



    def get_scores(self,query:torch.Tensor, key:torch.Tensor):
       #calculate scores btwn queries and keys
       return torch.einsum('ibhd,jbhd->ijbh,',query,key)
    
    # mask shape = [sequ_len_q, seq_len_k,batch_size]
    def prepare_mask(self, mask:torch.Tensor, query_shape:List[int], key_shape:List[int]):
       assert mask.shape[0] == 1 or mask.shape[0] == query_shape[0]
       assert mask.shape[1]==key_shape[0]
       assert mask.shape[2]==1 or mask.shape[2] ==query_shape[1]   

       mask=mask.unsqueeze(-1)
       return mask      

    # query key and value are the tensors that store collection of query, key and value vectors
    # They have shape [seq_len,batch_size,d_model].

    #mask shape= [seq_len,seq_len,batch_size] and mask[i,j,b] indicaates whether for batch b
    # query at position i has access to key-value at position j 

    def forward(self,*, query: torch.Tensor,
                key : torch.Tensor,
                value: torch.Tensor,
                mask: Optional[torch.Tensor]=None):
       seq_len , batch_size, _ = query.shape

       if mask is not None:
          mask=self.prepare_mask(mask,query.shape,key.shape)

          #prepare query , key and value for attention computation These will then have shape
          #[seq-len, batch_size, heads, d_k]
          query=self.query(query)
          key=self.key(key)
          value=self.value(value)

          scores=self.get_scores(query,key)

          scores *= self.scale

          if mask is not None:
             scores=scores.masked_fill(mask==0,float('-inf'))

             attn = self.softmax(scores)
             #save attentions if debugging 
             tracker.debug('attn',attn)
             attn=self.dropoutl(attn)
             x=torch.einsum("ijbh , jbhd->ibhd" , attn, value)
             self.attn = attn.detach()
             x=x.reshape(seq_len,batch_size,-1)
             return self.output(x)


tensor([1.9291, 0.1364], grad_fn=<ViewBackward0>)
